# The 2D Wavelength Solution

The book has produced two calibration products, and they have been used one after the other.
Chapter 6 fitted a **tilt solution**, mapping detector pixels into a rectified space where a
column is a single wavelength. Chapters 10 and 11 fitted a **1D wavelength solution** in that
rectified space, turning columns into Ångström.

Chained, those two describe something neither does alone: the wavelength of *any* pixel on the
raw detector. `specreduce.wavesol2d.WavelengthSolution2D` is that chain as an object.

Three things follow, and the chapter takes them in turn.

- **A wavelength solution for the whole frame.** `pix_to_wav(disp, cdisp)` answers "what
  wavelength is this pixel?" anywhere on the detector, and `wav_to_pix` inverts it. No
  rectification step, no assumption that a column is one wavelength.
- **Rectification and calibration in one step.** `resample` takes a raw detector frame straight
  to a wavelength grid, so background subtraction and extraction can be done *in wavelength
  space* rather than in pixel space.
- **A calibration that travels.** The combined solution is exposed as a GWCS object and
  serialized to ASDF with the chained mapping stored as a standalone GWCS, so it can be
  evaluated by code that is not `specreduce`. That is the main reason the object exists at
  present: it lets you bring your own extraction, for instance one that works on the raw
  detector pixels and avoids resampling altogether, and still have the wavelength of every
  pixel.

In [ ]:
import sys

sys.path.append("../src")

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import simple_norm
from specreduce.tilt_solution import TiltSolution
from specreduce.wavesol1d import WavelengthSolution1D
from specreduce.wavesol2d import WavelengthSolution2D

from common import (
    TILT_SOLUTION_FILE,
    WAVELENGTH_SOLUTION_FILE,
    fit_tilt_solution,
    fit_wavelength_solution,
    read_data,
)

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
ny, nx = obj.data.shape

## Combining the two solutions

Both components are on disk. Chapter 6 wrote the tilt solution and Chapter 10 the wavelength
solution, each as a few kilobytes of analytic model in an ASDF file. This is where that pays
off: the chapter that needs both reads both, and refits nothing.

In [ ]:
if not TILT_SOLUTION_FILE.exists():  # normally written by Chapter 6
    fit_tilt_solution(arcs).to_asdf(TILT_SOLUTION_FILE)
tilt = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

if not WAVELENGTH_SOLUTION_FILE.exists():  # normally written by Chapter 10
    fit_wavelength_solution(arcs, tilt).to_asdf(WAVELENGTH_SOLUTION_FILE)
wavelength = WavelengthSolution1D.from_asdf(WAVELENGTH_SOLUTION_FILE)

solution = WavelengthSolution2D(tilt, wavelength)

print(f"image shape   {solution.image_shape}")
print(f"dispersion    axis {solution.disp_axis}")
print(f"unit          {solution.unit}, air wavelengths: {solution.wave_air}")
print(f"pixel bounds  {solution.bounds_pix}")

That is the whole construction: two fitted objects in, one object out. The cross-dispersion
coordinate passes through the tilt transformation unchanged, so the chain is
`(disp, cdisp) → rectified column → wavelength`.

Note what `WavelengthSolution2D` needs to know that neither component carries on its own: the
**image shape**. The inverse tilt transformation is fitted numerically over the frame, so the
class has to know how big the frame is. Here it comes from the tilt solution's stored bounding
box; pass `image_shape=(ny, nx)` explicitly if your tilt solution does not have one.

## What wavelength is this pixel?

The point of the combined object is that this question now has an answer everywhere, not only in
a rectified frame.

In [ ]:
column = 512
rows = np.array([10, 70, 135, 250, 350])
wavelengths = np.array([float(solution.pix_to_wav(column, row)) for row in rows])

print(f"wavelength down column {column} of the raw detector:")
for row, value in zip(rows, wavelengths):
    print(f"  row {row:3d}:  {value:9.3f} A")
print(f"\nspread across the illuminated slit: {np.ptp(wavelengths):.2f} A")
print(f"in pixels, at this dispersion:      {np.ptp(wavelengths) / 5.29:.2f} pix")

**Thirty-two Ångström.** A single column of the raw detector spans that much wavelength between
the ends of the illuminated slit, about six pixels, and nearly fifty times the 0.66 Å rms of
the wavelength fit itself. This is the tilt of Chapter 6, restated in physical units, and it is
the clearest statement of why the assumption "a column is a wavelength" had to be dealt with
before anything else.

The inverse mapping is the more useful one in practice. Given a wavelength and a position along
the slit, where does it fall on the detector?

In [ ]:
line = 7635.106  # a bright ArI line from the GTC list

print(f"where {line} A lands, row by row:")
columns = np.array([float(solution.wav_to_pix(line, row)) for row in rows])
for row, value in zip(rows, columns):
    print(f"  row {row:3d}:  column {value:8.3f}")
print(f"\nspread: {np.ptp(columns):.2f} pixels")

The same 6 pixels, seen from the other side. That is the shape the arc lines had in Chapter 5's
figure, now written as a function you can evaluate rather than a distortion you have to look at.

`wav_to_pix` returns `nan` for wavelengths the solution does not cover, and both methods accept
masked arrays and propagate the mask, which is useful when the coordinates come from a detection
list that already carries one.

## Resampling the whole frame

`resample` takes a raw detector frame and bins every row independently onto a common wavelength
grid. Each row's bin edges are mapped back through the wavelength solution and the tilt
transformation into that row's own detector columns, and the enclosed flux is summed with
fractional weights at the edges.

The result is rectification and wavelength calibration in a single operation.

In [ ]:
calibrated = solution.resample(obj)

print(f"type              {type(calibrated).__name__}")
print(
    f"shape             {calibrated.flux.shape}, spectral axis index {calibrated.spectral_axis_index}"
)
print(f"flux unit         {calibrated.flux.unit}")
print(
    f"spectral axis     {calibrated.spectral_axis[0]:.1f} to {calibrated.spectral_axis[-1]:.1f}"
)
print(f"uncertainty       {type(calibrated.uncertainty).__name__}")
print(f"mask              {int(np.sum(calibrated.mask))} bins flagged")

A 2D `specutils` `Spectrum` whose spectral axis is wavelength, carrying a propagated variance and
a mask flagging any bin that drew on a masked detector pixel. The `spectral_axis_index` records
which axis is spectral, so downstream operations know the orientation without being told.

The flux is a **density**, in `electron / Angstrom`, on the same convention as
`WavelengthSolution1D.resample` (Chapter 10): the quantity conserved by the rebinning is the
integral over wavelength, not the sum of the values.

In [ ]:
edges = calibrated.spectral_axis.bin_edges.to(u.AA).value
integral = np.nansum(np.nan_to_num(calibrated.flux.value) * np.diff(edges)[None, :])

print(f"integral over wavelength: {integral:.6e} electron")
print(f"raw detector total:       {obj.data.sum():.6e} electron")
print(f"ratio:                    {integral / obj.data.sum():.6f}")

Flux is conserved to four parts in ten thousand, and the shortfall is the same edge loss the tilt
correction showed in Chapter 6: light that falls outside the wavelength grid at the corners of the
frame, where the most strongly tilted rows run off the end of the reference row's coverage.

Here is the calibrated frame, with the raw detector above it for comparison. The curves drawn on
the raw panel are **iso-wavelength contours**: each one traces a single wavelength across the
detector, computed with `wav_to_pix` at every row.

In [ ]:
fig, axs = plt.subplots(2, figsize=(8, 5.0), constrained_layout=True)

contour_wavelengths = np.arange(5500.0, 10001.0, 500.0)
rows_all = np.arange(float(ny))

axs[0].imshow(
    obj.data,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    norm=simple_norm(obj.data, stretch="log", vmin=0, vmax=250_000),
    extent=(0, nx - 1, 0, ny - 1),
)
for value in contour_wavelengths:
    columns = np.asarray(solution.wav_to_pix(value, rows_all), dtype=float)
    axs[0].plot(columns, rows_all, c="C1", lw=0.8, alpha=0.75)
    axs[0].text(
        np.nanmedian(columns),
        ny - 40,
        f"{value:.0f}",
        color="C1",
        size=6,
        ha="center",
        va="top",
        bbox=dict(fc="white", ec="none", alpha=0.6, pad=0.8),
    )
axs[0].set(
    xlim=(0, nx - 1),
    ylim=(0, ny - 1),
    xlabel="Dispersion [pix]",
    ylabel="Cross-dispersion [pix]",
)
axs[0].text(
    0.01,
    0.06,
    "raw detector, iso-wavelength contours [$\\AA$]",
    transform=axs[0].transAxes,
    va="bottom",
    size=8,
)

axis = calibrated.spectral_axis.to(u.AA).value
panel = np.nan_to_num(calibrated.flux.value)
axs[1].imshow(
    panel,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    norm=simple_norm(panel, stretch="log", vmin=0, vmax=250_000 / 5.29),
    extent=(axis[0], axis[-1], 0, ny - 1),
)
for value in contour_wavelengths:
    axs[1].axvline(value, c="C1", lw=0.8, alpha=0.75)
axs[1].set(
    xlim=(axis[0], axis[-1]),
    ylim=(0, ny - 1),
    xlabel=r"Wavelength [$\AA$]",
    ylabel="Cross-dispersion [pix]",
)
axs[1].text(
    0.01,
    0.06,
    "wavelength calibrated, same wavelengths",
    transform=axs[1].transAxes,
    va="bottom",
    size=8,
);

That figure is the chapter in one picture. The same ten wavelengths are curves on the detector and
straight vertical lines after resampling, which is the definition of what the operation does.

Two details are worth noticing. The contours **lean and bow**, rather than merely leaning: the
tilt is a 2D polynomial, not a shear, which is why Chapter 6 needed a surface fit rather than a
single angle. And the lean is **not constant with wavelength**: the 5500 Å contour spans about
7.2 pixels across the slit while the 10000 Å contour spans about 6.2. A correction fitted at one
end of the range and applied at the other would leave a residual of a pixel.

## Taking the solution elsewhere

`resample` allows a pipeline order the book has not used. Every chapter so far has rectified,
extracted, and then put the 1D spectrum on a wavelength axis; the combined solution allows the
reverse, calibrating the 2D frame first and extracting in wavelength space. On these data the two
orderings recover the same spectrum to **four parts in ten thousand**, with no trend across the
range, which is the end-to-end check on the whole chain: had the tilt, the wavelength solution,
or the chained resampling been wrong anywhere, the two would have parted company where the tilt
is largest. The ordering is therefore a matter of convenience. Calibrating first resamples once
where the book's order resamples twice, but neither extracts from independent pixels, since both
work on a resampled frame whose variances no longer describe independent samples.

The only extraction that keeps the detector's own statistics is one that runs on the raw frame,
and that is the case the combined solution is built for. Such a routine, whether it avoids
resampling on principle or models the instrument directly, needs only the wavelength of each
pixel it touches, and the solution supplies that to code outside `specreduce` in two forms. In
memory, the `gwcs` property is the chained detector-pixel-to-wavelength mapping as a GWCS
object. On disk, `to_asdf` writes the tilt and wavelength solutions separately, which is what
`from_asdf` reads back, and the chained mapping once more as a standalone GWCS that any
ASDF-aware tool can evaluate.

In [ ]:
combined_file = WAVELENGTH_SOLUTION_FILE.with_name("wavelength_solution_2d.asdf")
solution.to_asdf(combined_file)
restored = WavelengthSolution2D.from_asdf(combined_file)

# The same mapping as a GWCS object, for code that is not specreduce.
wcs = solution.gwcs
print(f"gwcs frames:  {wcs.input_frame.name} -> {wcs.output_frame.name}")
print(f"gwcs(512, 135) = {float(wcs(512, 135)):.3f} {solution.unit}")

## Summary

- **`WavelengthSolution2D` chains a tilt solution with a 1D wavelength solution** into the
  wavelength of any pixel on the raw detector. It is built from the two fitted objects, read back
  from their ASDF files with nothing refitted, and needs only the image shape besides, to fit the
  inverse tilt transformation.
- `pix_to_wav` and `wav_to_pix` answer "what wavelength is this pixel" and its inverse anywhere
  on the frame. A single detector column spans **32 Å**, about six pixels, between the ends of the
  illuminated slit: the tilt of Chapter 6 in physical units. Both accept masked arrays, and
  `wav_to_pix` returns `nan` outside the solution's coverage.
- `resample` goes from raw detector to a wavelength-calibrated 2D `Spectrum` in one step, with the
  variance propagated, a mask flagging bins fed by masked pixels, and the flux returned as a
  **density** whose integral over wavelength is conserved, to four parts in ten thousand here,
  the shortfall being edge loss at the corners of the frame.
- **The iso-wavelength contours**, `wav_to_pix` evaluated down every row, are the most direct
  picture of the distortion. They bow as well as lean, because the tilt is a 2D polynomial rather
  than a shear, and the lean changes with wavelength, from about 7.2 pixels across the slit at
  5500 Å to 6.2 at 10000 Å.
- **Pipeline order is a matter of convenience.** Extracting from the rectified frame and
  calibrating the 1D spectrum, or calibrating the 2D frame and extracting in wavelength space,
  give the same spectrum to 0.04 percent on these data, the end-to-end check on the whole chain.
  The second resamples once instead of twice, but both extract from a resampled frame.
- **The solution is built to travel.** Only extraction on the raw detector keeps independent
  pixel statistics, and a routine of your own that does so needs the wavelength of every raw
  pixel. The `gwcs` property supplies it in memory, and `to_asdf` stores the components
  separately plus the chained mapping as a standalone GWCS, so the solution can be evaluated
  without `specreduce`.

Next: [Flux Calibration](13-flux-calibration.ipynb), the one calibration step this book does
not yet perform.